# ARC-AGI-3: Process Journal

Registro del proceso de aprendizaje del agente. Anotaciones, decisiones, hallazgos.

**Objetivo:** Agente que resuelva juegos ARC-AGI-3 usando RSI + VSM.

**Stack:** LXC 114 (CPU) → Kaggle (GPU) → Competencia

---
## 2026-09-14: Session 1 — Foundation

### Contexto
- ARC-AGI-3: Competencia de puzzle solving con agentes interactivos
- Kaggle provee GPU (RTX 6000 96GB) para modelos grandes
- LXC 114: entorno de desarrollo local (4 cores, 8GB RAM)

### Decisiones tomadas
1. **Empezar con ls20** — juego de rotación de key (checkpoint → figure)
2. **Prompt mínimo** — solo vida, posición, objetivo (no contexto completo)
3. **Ollama local** — qwen3:1.7b para pruebas rápidas (2-3s/inference)
4. **Zone detection** — identificar regions clave del grid

### Hallazgos
- **Mecánica del juego:** Key rota cuando player pasa por checkpoint
- **Key fit:** Se detecta comparando distribución de colores (model vs figure)
- **Vida:** Baja 2 por movimiento (life_decreases_by_2)
- **Formas:** Model y figure son constantes (no cambian)

### Lo que funciona
- Zone detection: 4 zonas detectadas (model, figure, checkpoint, life_bar)
- Key-fit detection: Compara colores de model vs figure
- Prompt con objetivo: "Key doesn't fit. GO TO CHECKPOINT to rotate."
- Viewer: Bloques sólidos con caracteres (no colapsa)

### Lo que falta
- **Trajectory planning:** Calcular ruta más corta a checkpoint/figure
- **Budget tracking:** ¿La vida alcanza para la ruta?
- **Pathfinding:** ¿Cómo navegar el laberinto?
- **Strategy learning:** Aprender de trayectorias exitosas

### Próximos pasos
1. Implementar `arc3_trajectories.py` — guardar paths exitosos
2. Probar en otros juegos (vc33, ka59) — ¿cuál es más fácil?
3. Conectar a Kaggle GPU — probar con modelo grande

### Métricas
- Juegos probados: 1/25 (ls20)
- Niveles completados: 0
- Tiempo promedio por inference: 2-3s (CPU)
- Memoria usada: ~200MB / 8GB

---
## 2026-09-14: Session 1 — Trajectory Memory

### Contexto
- Necesitamos guardar trayectorias exitosas/failed para experience replay
- Cada intento del agente genera un trajectory con pasos, acciones, resultados
- Permite: recommend actions, find similar paths, learn from failures

### Implementación
- **arc3_trajectories.py** — TrajectoryMemory class
- Store: guarda trajectory en JSON (state/trajectories/)
- Query: get_successful, get_failed, get_best, get_action_stats
- Prompt fragment: recomienda acciones con alto success rate

### Integración con Agent
- Agent almacena trajectory al finalizar (GAME_OVER o WIN)
- Cada paso registrado: action, player_pos, life, key_fits, zones
- Prompt incluye memory context (acciones recomendadas)

### Primeros Resultados
- 1 trajectory almacenada: ls20, 1 step, failure (life 96→94)
- Agent ejecutando: 4 pasos, ACTION1 repetido (exploration bias)
- Life bajando 2 por movimiento (invariante confirmado)

### Próximos Pasos
1. Recomendar acciones no probadas (evitar ACTION1 repetido)
2. Analizar qué posiciones lead a checkpoint
3. Implementar pathfinding básico ( BFS a checkpoint/figure)

### Stack Actual
```
arc3_trajectories.py ← NUEVO (trajectory memory)
arc3_zones.py        ← Zone detection
arc3_memory.py       ← Pattern memory
arc3_feedback.py     ← Algedonic signals
arc3_canon.py        ← State drift
socratic_agent.py    ← UCB1 exploration
my_agent.py          ← Agent v2 (integrado)
```

---
## 2026-09-14: Session 1 — Pathfinding

### Contexto
- Agente necesitaba navegar el laberinto hacia checkpoint/figure
- Sin pathfinding, el agente exploraba aleatoriamente
- Necesitamos: distancia, costo, presupuesto, dirección

### Implementación
- **arc3_pathfinding.py** — BFS shortest path
- `identify_walkable()` — marca life bar como obstáculo
- `bfs()` — búsqueda en amplitud
- `plan_route()` — player → checkpoint → figure
- `to_prompt_fragment()` — incluir en prompt del LLM

### Mecánica
1. Detectar celdas caminables (excluir life bar)
2. BFS desde player hasta checkpoint
3. BFS desde checkpoint hasta figure
4. Calcular costo total: distancia × 2
5. Verificar presupuesto: vida >= costo

### Resultados
- Agent ve: "Pathfinding: go_to_checkpoint, distance=24, cost=48, budget_ok=True"
- Fallback: pathfinding → socratic → random
- Primer test: agent开始 a planificar ruta

### Stack Actual
```
arc3_pathfinding.py  ← NUEVO (BFS pathfinding)
arc3_trajectories.py ← Trajectory memory
arc3_zones.py        ← Zone detection
arc3_memory.py       ← Pattern memory
arc3_feedback.py     ← Algedonic signals
arc3_canon.py        ← State drift
socratic_agent.py    ← UCB1 exploration
my_agent.py          ← Agent v2 (integrado)
```

---
## 2026-09-15: Multi-Game Test

### Resultados

| Game | Grid | Colors | Zones Detected | LLM Response | Status |
|---|---|---|---|---|---|
| ls20 | 64x64 | 10 | model, figure, checkpoint, life_bar | ✅ "GO TO CHECKPOINT" | **Working** |
| vc33 | 64x64 | 7 | figure, checkpoint | ✅ Response OK | Partial |
| ka59 | 64x64 | 7 | None | ⚠️ No zone data | Needs work |
| sk48 | 64x64 | 10 | None | ⚠️ No zone data | Needs work |
| r11l | 64x64 | 7 | model, figure | ✅ "Key doesn't fit" | Partial |

### Análisis
- **ls20**: Mejor funcionando — todas las zonas detectadas, pathfinding activo
- **r11l**: Detecta model + figure pero no checkpoint — posible layout diferente
- **vc33**: Detecta figure + checkpoint pero no model — necesita ajuste
- **ka59/sk48**: No detecta zonas — colores diferentes a ls20

### Próximos Pasos
1. **Ajustar zone detection** para ka59/sk48 (colores diferentes)
2. **Analizar grids** de cada juego para entender layout
3. **Enfocar en ls20** — es el que mejor funciona
4. **Probar con más steps** — ver si agent llega a checkpoint

---
## 2026-09-15: Critical Bug Fix — Player Detection

### Problema
- Player no se movía en 12 pasos
- Zone detector reportaba posición (8, 32) — equivocada
- Pathfinding enviaba ACTION2 (down) — dirección incorrecta

### Causa Raíz
1. **Player detection buscaba celdas verdes (color 3)**
   - Hay 892 celdas verdes en el grid — el player NO es verde
   - El player es un sprite 5x5: filas 0-1=color 12 (rose), filas 2-4=color 9 (brown)
   - Posición real: **(45, 34)**, no (8, 32)

2. **Walkable detection no identificaba paredes**
   - Celdas amarillas (color 4) = paredes, NO caminables
   - El player estaba rodeado de muros — no podía bajar
   - Pathfinding decía ACTION2 (down) pero había pared

### Fixes
- `_detect_player()`: ahora busca patrón 5x5 (12+9) con fallback a verde
- `identify_walkable()`: amarillo (4) = pared, NO caminable
- Pathfinding ahora calcula ruta correcta

### Resultado
- Player se movió: (45, 34) → (27, 36) → navegando hacia checkpoint (32, 31)
- Mezcla de ACTION1 (up) y ACTION2 (down) — walkthrough del laberinto
- Life bajando: 96 → 76 (20 en 11 pasos)

### Lección Aprendida
**Siempre verificar la detección del player contra el grid real.**
El asumir "verde = player" causó 12 pasos de movimiento fantasma.

---
## 2026-09-15: Game Mechanics Deep Dive

### Step Counter (Life Real)
- **StepCounter: 42** (inicial por nivel)
- **StepsDecrement: 1** (por movimiento exitoso)
- Cuando llega a 0 → `self.lose()` → GAME_OVER
- La "vida" que vemos es visual — el real es 42 pasos

### Sprite Rotation
- El sprite del player **rota** cuando pasa por el checkpoint
- Antes: 5x5 con colores 12 (rose) + 9 (brown)
- Después de rotación: patrón cambia
- Fix: detectar cualquier 5x5 con 2 colores único en el grid

### Game Cycle
- ANTES: step limit fijo (20)
- DESPUÉS: corre hasta GAME_OVER o WIN (safety 500)
- El juego termina naturalmente con el step counter

### Parsed Code Insights
- `self.aqygnziho` = step counter (42)
- `self.lose()` = game over when counter = 0
- `self.next_level()` = advance when figure reached
- `self.gudziatsk` = player sprite
- `self.srgbthxut` = key/model sprites
- `self.plrpelhym` = figure/lock sprites

---
## 2026-09-14: Session 1 — Architecture

### Stack VSM (Recursive)

```
S5 (Política): current_focus.json + git commits
  ↓
S4 (Inteligencia): socratic_agent.py + arc3_memory.py
  ↓
S3 (Control): arc3_canon.py + arc3_feedback.py
  ↓
S2 (Coordinación): arc3_zones.py + my_agent.py
  ↓
S1 (Operaciones): viewer.py + run.py + arc-agi-3 API
```

### Capas de Aprendizaje (Proyectadas)

```
Nivel 1: Memoria de Trayectorias
  arc3_trajectories.py → guardar paths exitosos

Nivel 2: Abstracción de Estrategia
  arc3_strategy.py → templates por tipo de juego

Nivel 3: Curriculum Pedagógico
  arc3_curriculum.py → progresión de dificultad

Nivel 4: Transferencia
  arc3_transfer.py → aplicar conocimiento entre juegos
```

### Metodologías Elegidas

1. **Scaffolded Exploration** (Vygotsky)
   - Empezar con estructuras simples
   - Progresar a complejidad
   - Hints progresivos

2. **Experience Replay** (DQN)
   - Guardar cada intento
   - Reentrenar con experiencias pasadas

3. **Curiosity-Driven Exploration**
   - Buscar estados nuevos
   - Explorar zonas no visitadas

4. **Hierarchical RL**
   - Sub-objetivos: checkpoint → key rotation → figure
   - Planificación por niveles

---
## Key Metrics Dashboard

| Métrica | Valor | Target |
|---|---|---|
| Juegos resueltos | 0/25 | 1 |
| Niveles completados | 0 | 1 |
| Tiempo por inference | 2-3s | <1s |
| Memoria usada | 200MB | <2GB |
| Patrones aprendidos | 59 | 100+ |
| Trayectorias guardadas | 0 | 50+ |

---
## Decisions Log

| Fecha | Decisión | Razón | Resultado |
|---|---|---|---|
| 2026-09-14 | Empezar con ls20 | Key rotation es mecánica clara | Pendiente |
| 2026-09-14 | Prompt mínimo | LLM no necesita contexto completo | Funciona |
| 2026-09-14 | Ollama local | Pruebas rápidas sin GPU | 2-3s OK |
| 2026-09-14 | Zone detection | Spatial awareness es clave | 4 zonas detectadas |